In [ ]:
#
# ⚡ UNIVERSAL FIRST CELL - Run this FIRST in every notebook!
# Compatible with: Google Colab, GitHub Codespaces, Local
#

import os
import subprocess
import sys

# Detect environment
IS_COLAB = "google.colab" in sys.modules
IS_CODESPACES = os.path.exists("/.devcontainer") or os.path.exists("/workspaces")
IS_LOCAL = not (IS_COLAB or IS_CODESPACES)

print(f"\U0001f680 Environment: {'Colab' if IS_COLAB else 'Codespaces' if IS_CODESPACES else 'Local'}")

# Ensure we are in the root directory for relative paths
while not os.path.exists('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print(f"\U0001f4c1 Working directory set to: {os.getcwd()}")

# Install BQ dependency on Colab
if IS_COLAB:
    !pip install google-cloud-bigquery -q

print("\u2705 Environment ready!")

# 12 - GDELT Domain-Filtered Buildout Mining

**Goal**: Query GDELT v2 GKG by domain (datacenterdynamics.com, datacenterknowledge.com, siliconangle.com) × 20 DC companies to discover data center buildout announcements.

**Approach**: Domain-filtered GKG query (no broad theme matching) → ~1-5 GB vs 148 GB from previous broad approach.

**Output**: `data/raw/buildout_candidates_gkg.csv`

In [ ]:
import pandas as pd
import numpy as np
import json
import warnings
from datetime import datetime
from pathlib import Path
warnings.filterwarnings('ignore')

from google.cloud import bigquery

print("\u2705 Libraries imported")

## Step 1: BigQuery Authentication

In [ ]:
# BigQuery authentication
# On Colab: GOOGLE_APPLICATION_CREDENTIALS = /content/gcp_adc.json
# Local: Use gcloud auth application-default login

PROJECT = os.environ.get("GOOGLE_CLOUD_PROJECT", "")

# Check for credentials file if env var not set
if not os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"):
    for cred_path in ["/content/gcp_adc.json", "docs/gcp_adc.json"]:
        if os.path.exists(cred_path):
            os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = cred_path
            print(f"\U0001f4c4 Using credentials from: {cred_path}")
            break

if not os.environ.get("GOOGLE_APPLICATION_CREDENTIALS"):
    print("\u274c GOOGLE_APPLICATION_CREDENTIALS not set and no creds file found.")
    print("   On Colab: Upload gcp_adc.json to /content/")
    print("   Local: Run 'gcloud auth application-default login'")
    print("   See docs/gcp_adc.json (not tracked by git)")
    raise SystemExit(0)

# Try to extract project ID from credentials file
if not PROJECT:
    try:
        with open(os.environ["GOOGLE_APPLICATION_CREDENTIALS"]) as f:
            creds = json.load(f)
            PROJECT = creds.get("project_id", "")
    except Exception as e:
        print(f"\u26a0\ufe0f Could not read credentials: {e}")

if not PROJECT:
    print("\u274c Could not determine GCP project ID from credentials or env.")
    print("   Set GOOGLE_CLOUD_PROJECT env var or add project_id to your ADC JSON.")
    raise SystemExit(0)

# Create client
client = bigquery.Client(project=PROJECT)
print(f"\u2705 BQ client ready (project={PROJECT})")

# Verify: list datasets to confirm auth works
print("\U0001f4ca Verifying access...")
datasets = list(client.list_datasets())
print(f"   Accessible datasets: {len(datasets)}")
for ds in datasets:
    print(f"   - {ds.dataset_id}")
print("\u2705 BigQuery authentication verified!")

## Step 2: Define Query

Domain-filtered GKG query. Only 3 trusted DC industry domains. No broad theme matching.

**Schema**: `gdelt-bq.gdeltv2.gkg_partitioned`
- `SourceCommonName`: domain name (our filter)
- `V2Organizations`: pipe-separated `Name,Count;Name,Count`
- `V2Locations`: location mentions
- `DocumentIdentifier`: article URL
- `V2Tone`: tonal score

In [ ]:
# Target companies and their name variants (as they appear in V2Organizations)
# Covers all 20 tickers from the buildout promises dataset
COMPANY_KEYWORDS = [
    # Hyperscalers
    'Microsoft',
    'Google', 'Alphabet',
    'Amazon', 'AWS',
    'Meta', 'Facebook',
    'NVIDIA', 'Nvidia',
    'Apple',
    # Enterprise cloud
    'Oracle',
    # GPU cloud
    'Crusoe',
    # Colo / digital infra REITs
    'Equinix',
    'Digital Realty',
    'American Tower',
    'Prologis',
    'Simon Property',
    'Public Storage',
    'Outfront',
    'Sabra',
    'Hudson Pacific',
    'Rexford',
    'First Industrial',
    'SITC',
]

print(f"\U0001f3af Tracking {len(COMPANY_KEYWORDS)} company name patterns")
for kw in COMPANY_KEYWORDS:
    print(f"   - {kw}")

# Date range: 2020-01-01 to present
start_date = '2020-01-01'
end_date = datetime.now().strftime('%Y-%m-%d')
print(f"\U0001f4c5 Date range: {start_date} to {end_date}")

# Build SQL with LIKE conditions for each company keyword
org_conditions = ' OR '.join(
    f"V2Organizations LIKE '%{kw}%'" for kw in COMPANY_KEYWORDS
)

sql = f"""
SELECT
  DATE, SourceCommonName, DocumentIdentifier,
  V2Organizations, V2Locations, V2Tone
FROM `gdelt-bq.gdeltv2.gkg_partitioned`
WHERE _PARTITIONTIME >= TIMESTAMP("{start_date}")
  AND SourceCommonName IN (
    'datacenterdynamics.com',
    'datacenterknowledge.com',
    'siliconangle.com'
  )
  AND ({org_conditions})
ORDER BY DATE DESC
"""

print(f"\u2705 SQL query defined ({len(sql.split())} words)")

## Step 3: Dry Run (Cost Estimate)

Estimate bytes processed before running the actual query. Budget: <5 GB.

In [ ]:
# Dry run to estimate cost
MAX_GB = 600  # ~$3 at $5/TB for 2.5yr GKG scan
COST_PER_TB = 5  # $5 per TB for BQ on-demand

print("\U0001f50d Running dry run...")
job_config = bigquery.QueryJobConfig(dry_run=True, use_query_cache=False)
dry_run_job = client.query(sql, job_config=job_config)
bytes_processed = dry_run_job.total_bytes_processed
gb_processed = bytes_processed / 1e9
cost_usd = gb_processed / 1000 * COST_PER_TB

print(f"   Data processed: {bytes_processed:,} bytes ({gb_processed:.2f} GB)")
print(f"   Estimated cost: ${cost_usd:.2f} at ${COST_PER_TB}/TB")

if gb_processed > MAX_GB:
    print(f"\u274c WARNING: Estimated cost {gb_processed:.2f} GB exceeds {MAX_GB} GB limit!")
    print("   Reduce date range in the SQL query above and re-run.")
    print("   Aborting to avoid unexpected charges.")
    raise SystemExit(0)
else:
    print(f"\u2705 Estimated ${cost_usd:.2f} -- proceeding to execute...")


## Step 4: Execute Query

In [ ]:
# Execute the query
print("\U0001f504 Executing query...")
query_job = client.query(sql)
df = query_job.to_dataframe()

bytes_billed = int(query_job.total_bytes_billed or 0)
print(f"\u2705 Query returned {len(df)} rows")
print(f"\U0001f4b0 Bytes billed: {bytes_billed:,} ({bytes_billed/1e9:.2f} GB)")

if df.empty:
    print("\u274c No results returned. Check query filters.")
    print("   Possible reasons:")
    print("   - Domain names may not match exactly (check case)")
    print("   - Company names may differ in V2Organizations format")
    print("   - No articles from these domains in the date range")
    raise SystemExit(0)

# Display sample
print(f"\n\U0001f4cb First 5 rows:")
print(df[['DATE', 'SourceCommonName', 'V2Organizations']].head(5).to_string(index=False))

## Step 5: Summary Statistics

In [ ]:
# --- Per-domain count ---
print("\U0001f4ca Per-domain count:")
print(df['SourceCommonName'].value_counts().to_string())
print()

# --- Parse companies from V2Organizations ---
def find_matching_companies(org_str):
    """Find which of our target companies appear in V2Organizations."""
    found = []
    for kw in COMPANY_KEYWORDS:
        if kw.lower() in str(org_str).lower():
            found.append(kw)
    return found if found else ['unknown']

df['matched_companies'] = df['V2Organizations'].apply(find_matching_companies)
df_exploded = df.explode('matched_companies')

print("\U0001f4ca Per-company mention count (from V2Organizations):")
company_counts = df_exploded['matched_companies'].value_counts()
print(company_counts.to_string())
print()

# --- Per-year count ---
df['year'] = df['DATE'].astype(str).str[:4]
print("\U0001f4ca Per-year count:")
year_counts = df['year'].value_counts().sort_index()
print(year_counts.to_string())
print()

# --- Total ---
print(f"\U0001f4c8 Total candidates: {len(df)}")

## Step 6: Save Output

Save candidates CSV and track with DVC.

In [ ]:
# Ensure output directory exists
os.makedirs('data/raw', exist_ok=True)

# Save to CSV
output_path = 'data/raw/buildout_candidates_gkg.csv'
df.to_csv(output_path, index=False)
print(f"\U0001f4be Saved {len(df)} candidates to {output_path}")
print(f"   File size: {os.path.getsize(output_path):,} bytes")

# DVC tracking
print("\n\U0001f504 Running DVC add...")
try:
    import subprocess
    result = subprocess.run(
        ['dvc', 'add', output_path],
        capture_output=True, text=True, check=True
    )
    print(result.stdout)
    
    # Push to remote
    result_push = subprocess.run(
        ['dvc', 'push', output_path + '.dvc'],
        capture_output=True, text=True
    )
    if result_push.returncode == 0:
        print("\u2705 DVC push successful")
    else:
        print(f"\u26a0\ufe0f DVC push issue (may need remote config): {result_push.stderr}")
except Exception as e:
    print(f"\u26a0\ufe0f DVC step skipped: {e}")
    print("   Run 'dvc add data/raw/buildout_candidates_gkg.csv' manually.")

print(f"\n\u2705 Notebook complete! Output ready for article extraction (Task 5).")

## Summary

✅ **GDELT Domain-Filtered Buildout Mining Complete**

- Queried GDELT v2 GKG for 3 DC industry domains × 20 company patterns
- Domain filter (no broad theme matching) kept costs under 5 GB
- Output: `data/raw/buildout_candidates_gkg.csv`
- DVC tracked

**Next steps**:
- Run `notebooks/13-article-extraction.ipynb` to fetch and parse article text
- Extract: MW, location, target dates from article content
- Cross-reference with gridstatus ISO queue for promise_kept labeling